# Analyzing Images with Azure AI Vision

In this lab, you will use the Azure AI Vision Python SDK to run a full image analysis pipeline — tags, captions, dense captions, object detection, people detection, and OCR — across multiple sample images.

## Learning Objectives

- Configure the `ImageAnalysisClient` from the `azure-ai-vision-imageanalysis` SDK
- Request multiple visual features in a single analysis call
- Extract and display tags, captions, dense captions, objects, people, and OCR results
- Apply confidence thresholds to filter results
- Analyze multiple images and compare results across different content types

## Prerequisites

- An active Azure subscription
- A **multi-service Azure AI Services (Foundry)** resource deployed in a region that supports Image Analysis 4.0 (for example, **East US**; caption, dense captions, and people detection are region-gated). This single resource provides the Vision endpoint — **no separate Azure AI Vision resource is required**.
- **Keyless authentication with Microsoft Entra ID (RBAC).** Newer Foundry resources ship with API keys disabled, so this lab authenticates with your Azure identity instead of a key. Two setup steps:
  1. Grant your identity the **Cognitive Services User** role on the resource. Being subscription *Owner* is **not** enough — that covers management (control plane), not the data-plane calls the SDK makes.
     ```bash
     # your object id:   az ad signed-in-user show --query id -o tsv
     # the resource id:  az cognitiveservices account show -n <resource> -g <rg> --query id -o tsv
     az role assignment create \
       --assignee-object-id <your-object-id> --assignee-principal-type User \
       --role "Cognitive Services User" \
       --scope <resource-id>
     ```
  2. Sign in so `DefaultAzureCredential` can obtain a token: `az login`
- Your resource endpoint (`https://<resource>.cognitiveservices.azure.com/`) in `.env` (copy `.env.example`); leave `AZURE_AI_VISION_KEY` blank to use Entra ID.

## 1 — Install Dependencies

In [ ]:
%pip install azure-ai-vision-imageanalysis azure-identity python-dotenv --quiet

## 2 — Load Configuration

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

ENDPOINT = os.environ["AZURE_AI_VISION_ENDPOINT"]
KEY = os.environ.get("AZURE_AI_VISION_KEY", "").strip()
if KEY.startswith("your-") or KEY.startswith("<"):
    KEY = ""

print(f"Endpoint: {ENDPOINT}")
print(f"Authentication: {'API key' if KEY else 'DefaultAzureCredential'}")


## 3 — Create the ImageAnalysisClient

The `ImageAnalysisClient` is the entry point for all SDK-based image analysis operations. It uses `DefaultAzureCredential` when no API key is configured.

In [ ]:
from azure.ai.vision.imageanalysis import ImageAnalysisClient
from azure.ai.vision.imageanalysis.models import VisualFeatures
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

credential = AzureKeyCredential(KEY) if KEY else DefaultAzureCredential()
client = ImageAnalysisClient(
    endpoint=ENDPOINT,
    credential=credential,
)

print("ImageAnalysisClient created successfully")


## 4 — Define Sample Images

We analyze two local images from the `Images/` folder, each exercising different capabilities:

- **Microsoft Redmond** (`Microsoft Redmond.jpg`) — a campus building with the "Microsoft" sign; good for tags, caption, objects, and OCR of the sign text.
- **Underwater Datacenter** (`Underwater Datacenter.jpg`) — the Project Natick team standing in front of the sealed datacenter capsule; good for people detection, objects, caption, and tags.

Because the images live on disk, we read each file as bytes and pass them to `client.analyze()` (instead of `analyze_from_url`). The `load_image()` helper below does the read.

In [ ]:
from pathlib import Path

IMAGES_DIR = Path("Images")

SAMPLE_IMAGES = {
    "redmond": IMAGES_DIR / "Microsoft Redmond.jpg",
    "underwater_datacenter": IMAGES_DIR / "Underwater Datacenter.jpg",
}


def load_image(name):
    """Read a sample image file as bytes for client.analyze()."""
    return SAMPLE_IMAGES[name].read_bytes()


print(f"Loaded {len(SAMPLE_IMAGES)} sample image files")
for name, path in SAMPLE_IMAGES.items():
    status = "OK" if path.exists() else "MISSING"
    size = path.stat().st_size if path.exists() else 0
    print(f"  {name:22s} {status:8s} {size:>9,} bytes  {path}")

## 5 — Full Analysis: Tags, Caption, Objects, and People

Request all major visual features in a single call against the **Microsoft Redmond** image. We read the file as bytes and call `client.analyze()`, passing a list of `VisualFeatures` enum values.

In [ ]:
result = client.analyze(
    image_data=load_image("redmond"),
    visual_features=[
        VisualFeatures.TAGS,
        VisualFeatures.CAPTION,
        VisualFeatures.DENSE_CAPTIONS,
        VisualFeatures.OBJECTS,
        VisualFeatures.PEOPLE,
        VisualFeatures.READ,
    ],
)

print(f"Image dimensions: {result.metadata.width} x {result.metadata.height}")
print(f"Model version: {result.model_version}")
print()

# Quick summary of everything returned by this single call — the sections below drill into each.
caption = result.caption.text if result.caption else "—"
n_tags = len(result.tags.list) if result.tags else 0
n_dense = len(result.dense_captions.list) if result.dense_captions else 0
n_objects = len(result.objects.list) if result.objects else 0
n_people = len(result.people.list) if result.people else 0
n_lines = sum(len(block.lines) for block in result.read.blocks) if result.read else 0

print(f'Caption: "{caption}"')
print(
    f"Returned in one call:  {n_tags} tags · {n_dense} dense captions · "
    f"{n_objects} objects · {n_people} people · {n_lines} text lines"
)

## 6 — Display Tags

Tags are content labels drawn from a taxonomy of over 3,000 categories. Each tag has a confidence score.

In [ ]:
CONFIDENCE_THRESHOLD = 0.7

if result.tags:
    print(f"Tags ({len(result.tags.list)} total, showing >= {CONFIDENCE_THRESHOLD} confidence):\n")
    for tag in result.tags.list:
        if tag.confidence >= CONFIDENCE_THRESHOLD:
            print(f"  {tag.name:30s}  {tag.confidence:.4f}")
else:
    print("No tags returned")

## 7 — Display Caption and Dense Captions

The caption provides a single sentence describing the whole image. Dense captions describe individual regions, each with its own bounding box.

In [ ]:
# Single caption
if result.caption:
    print(f"Caption: \"{result.caption.text}\"")
    print(f"  Confidence: {result.caption.confidence:.4f}")
    print()

# Dense captions
if result.dense_captions:
    print(f"Dense captions ({len(result.dense_captions.list)} regions):\n")
    for dc in result.dense_captions.list:
        bbox = dc.bounding_box
        print(f"  \"{dc.text}\"")
        print(f"    Confidence: {dc.confidence:.4f}")
        print(f"    Region: x={bbox.x}, y={bbox.y}, w={bbox.width}, h={bbox.height}")
        print()

## 8 — Display Object Detection Results

Each detected object includes a label, confidence score, and bounding box in pixel coordinates.

In [ ]:
if result.objects:
    print(f"Objects detected: {len(result.objects.list)}\n")
    for obj in result.objects.list:
        for tag in obj.tags:
            bbox = obj.bounding_box
            print(f"  {tag.name:20s}  confidence: {tag.confidence:.4f}")
            print(f"    Bounding box: x={bbox.x}, y={bbox.y}, w={bbox.width}, h={bbox.height}")
else:
    print("No objects detected")

## 9 — Display People Detection Results

People detection returns bounding boxes for each person in the image. This is separate from object detection and provides higher accuracy for person-specific scenarios.

In [ ]:
if result.people:
    print(f"People detected: {len(result.people.list)}\n")
    for person in result.people.list:
        bbox = person.bounding_box
        print(f"  Confidence: {person.confidence:.4f}")
        print(f"    Bounding box: x={bbox.x}, y={bbox.y}, w={bbox.width}, h={bbox.height}")
else:
    print("No people detected")

## 10 — Display OCR Results

The Read feature extracts printed and handwritten text. Results are organized as blocks, lines, and words — each word with a bounding polygon.

In [ ]:
if result.read:
    blocks = result.read.blocks
    print(f"Text blocks: {len(blocks)}\n")
    for block_idx, block in enumerate(blocks):
        print(f"Block {block_idx + 1}:")
        for line in block.lines:
            word_confidences = [w.confidence for w in line.words]
            avg_conf = sum(word_confidences) / len(word_confidences) if word_confidences else 0
            print(f"  \"{line.text}\"  (avg confidence: {avg_conf:.4f})")
        print()
else:
    print("No text detected")

## 11 — Extract Printed Text with Word-Level Geometry

The Read feature extracts text (printed and handwritten) and returns it as blocks, lines, and words. Here we run a focused OCR pass on the **Microsoft Redmond** image — which contains the building's "Microsoft" sign and address number — and inspect the per-word bounding polygon to see the geometry the API returns for each word.

In [ ]:
ocr_result = client.analyze(
    image_data=load_image("redmond"),
    visual_features=[VisualFeatures.READ],
)

if ocr_result.read and ocr_result.read.blocks:
    print("Printed text extracted:\n")
    for block in ocr_result.read.blocks:
        for line in block.lines:
            print(f"  \"{line.text}\"")
            # Show the bounding polygon for the first word to illustrate the format
            if line.words:
                first_word = line.words[0]
                polygon_points = first_word.bounding_polygon
                print(f"    First word polygon: {[(p.x, p.y) for p in polygon_points]}")
else:
    print("No text detected")

## 12 — Analyze People in a Group Scene

Let us run caption, people, and object detection together on the **Underwater Datacenter** image, which shows the Project Natick team in front of the datacenter capsule. People detection returns a bounding box per person and is separate from (and more accurate than) treating "person" as an object label.

In [ ]:
people_result = client.analyze(
    image_data=load_image("underwater_datacenter"),
    visual_features=[
        VisualFeatures.CAPTION,
        VisualFeatures.PEOPLE,
        VisualFeatures.OBJECTS,
    ],
)

if people_result.caption:
    print(f"Caption: \"{people_result.caption.text}\"")
    print(f"  Confidence: {people_result.caption.confidence:.4f}\n")

if people_result.people:
    people = people_result.people.list
    print(f"People detected: {len(people)}")
    for i, person in enumerate(people):
        bbox = person.bounding_box
        print(f"  Person {i + 1}: confidence={person.confidence:.4f}, "
              f"bbox=({bbox.x}, {bbox.y}, {bbox.width}, {bbox.height})")

print()

if people_result.objects:
    print(f"Objects detected: {len(people_result.objects.list)}")
    for obj in people_result.objects.list:
        for tag in obj.tags:
            print(f"  {tag.name}: confidence={tag.confidence:.4f}")

## 13 — Compare Results Across Images

Run tag extraction on all sample images and compare the top tags to see how the model adapts to different content types.

In [ ]:
for name in SAMPLE_IMAGES:
    tag_result = client.analyze(
        image_data=load_image(name),
        visual_features=[VisualFeatures.TAGS, VisualFeatures.CAPTION],
    )

    print(f"\n--- {name} ---")
    if tag_result.caption:
        print(f"Caption: \"{tag_result.caption.text}\"")

    print("Top 5 tags:")
    if tag_result.tags:
        top_tags = sorted(
            tag_result.tags.list, key=lambda t: t.confidence, reverse=True
        )[:5]
        for tag in top_tags:
            print(f"  {tag.name:25s} {tag.confidence:.4f}")
    else:
        print("  (none)")

## Summary

In this lab you:

- Configured the `ImageAnalysisClient` with an Azure AI Vision endpoint and credential
- Requested tags, captions, dense captions, objects, people, and OCR in a single analysis call
- Analyzed **local image files** by reading them as bytes and calling `client.analyze()`
- Extracted and displayed structured results for each visual feature
- Applied confidence thresholds to filter tags
- Extracted printed text and examined word-level bounding polygon geometry
- Compared analysis results across images with different content types

The SDK abstracts the REST API into typed Python objects. You get the same results as the REST approach but with better type safety and less boilerplate.